# Hybrid Retrieval Lab: Build the Authorized Pipeline in PyTorch

Part 01a proved why Riverside needs complementary retrieval signals. This lab resumes the
same six versioned passages and builds the complete path:

`identity -> ACL filter -> BM25 + PyTorch dense -> fusion -> rerank -> support gate -> telemetry`

**Deliverable:** a per-query result, degraded-mode record, and verified index manifest.
Nothing downloads from the network; the tiny semantic encoder exposes tensors and state.


## Learning route

| Part | Build | Failure contract |
| --- | --- | --- |
| 1 | BM25 and `nn.Module` dense encoder | Exact code and paraphrase both recover |
| 2 | RRF and weighted fusion | Tune `k`/`alpha` on labels, not anecdotes |
| 3 | PyTorch pair reranker | Membership stays fixed; only rank changes |
| 4 | ACL and support gates | Fail closed and abstain |
| 5 | Degraded mode and telemetry | Never disguise fallback as healthy |
| 6 | Immutable manifest | Dense and sparse artifacts publish atomically |


In [ ]:
# Imports, deterministic state, and shared fixtures.
import hashlib
import json
import math
import tempfile
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

from rag_shared import (
    COURSE_CAVEAT, DOCUMENTS, RETRIEVAL_QUERIES, SEMANTIC_FEATURES,
    assert_fixture_contracts, authorized_documents, set_seed, tokenize,
)

set_seed()
assert_fixture_contracts()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={DEVICE}; documents={len(DOCUMENTS)}")
print(COURSE_CAVEAT)


## Part 1 - Build both first-stage retrievers

BM25 owns exact strings. The dense encoder owns a tiny inspectable semantic space. The
encoder contains a fixed embedding table registered as state, runs in evaluation mode, and
emits `[batch, semantic_features]` float tensors.


In [ ]:
# Manual BM25: visible term frequency, inverse document frequency, and length normalization.
class BM25Index:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.documents = documents
        self.k1, self.b = k1, b
        self.tokens = [tokenize(document["text"]) for document in documents]
        self.avg_length = np.mean([len(row) for row in self.tokens])
        frequency = Counter(token for row in self.tokens for token in set(row))
        self.idf = {
            token: math.log(1 + (len(documents) - count + 0.5) / (count + 0.5))
            for token, count in frequency.items()
        }

    def scores(self, query):
        values = []
        for row in self.tokens:
            counts = Counter(row)
            score = 0.0
            for token in tokenize(query):
                tf = counts[token]
                denominator = tf + self.k1 * (1 - self.b + self.b * len(row) / self.avg_length)
                score += self.idf.get(token, 0.0) * tf * (self.k1 + 1) / max(denominator, 1e-9)
            values.append(score)
        return np.asarray(values, dtype=np.float64)


**Walkthrough:** the index stores tokenized data and corpus statistics, not learned weights.
`scores` returns one float per authorized document. The lab deliberately avoids an opaque
retriever wrapper so the BM25 state and length correction remain visible.


In [ ]:
# Tiny PyTorch semantic encoder with explicit buffer state and tensor shapes.
class ConceptEncoder(nn.Module):
    def __init__(self, token_to_feature):
        super().__init__()
        self.features = sorted(set(token_to_feature.values()))
        self.feature_index = {name: index for index, name in enumerate(self.features)}
        self.token_to_feature = token_to_feature
        self.register_buffer("identity", torch.eye(len(self.features)))

    def forward(self, texts):
        rows = []
        for text in texts:
            vector = torch.zeros(len(self.features), device=self.identity.device)
            for token in tokenize(text):
                feature = self.token_to_feature.get(token)
                if feature:
                    vector += self.identity[self.feature_index[feature]]
            rows.append(vector)
        batch = torch.stack(rows)
        return torch.nn.functional.normalize(batch, dim=1)

dense_encoder = ConceptEncoder(SEMANTIC_FEATURES).to(DEVICE).eval()
with torch.no_grad():
    document_embeddings = dense_encoder([document["text"] for document in DOCUMENTS])
print("embedding shape:", tuple(document_embeddings.shape))
print("dtype/device:", document_embeddings.dtype, document_embeddings.device)
assert document_embeddings.shape == (len(DOCUMENTS), len(dense_encoder.features))


In [ ]:
# Run both retrievers against the same authorized candidate list.
def first_stage(query, groups):
    candidates = authorized_documents(groups)
    bm25 = BM25Index(candidates)
    lexical = bm25.scores(query)
    with torch.no_grad():
        query_vector = dense_encoder([query])
        candidate_vectors = dense_encoder([document["text"] for document in candidates])
        dense = (query_vector @ candidate_vectors.T).squeeze(0).cpu().numpy()
    return candidates, lexical, dense

evidence_rows = []
for case in RETRIEVAL_QUERIES[:2]:
    candidates, lexical, dense = first_stage(case["question"], case["groups"])
    for method, scores in [("BM25", lexical), ("dense", dense)]:
        order = np.argsort(-scores, kind="stable")
        ranked_ids = [candidates[index]["id"] for index in order]
        evidence_rows.append({
            "query": case["kind"], "method": method, "top": ranked_ids[0],
            "gold_rank": next(i for i, doc_id in enumerate(ranked_ids, 1) if doc_id in case["gold_ids"]),
        })
display(pd.DataFrame(evidence_rows))


## Part 2 - Fuse without pretending the score units match

RRF uses ranks. Weighted fusion first normalizes each score vector. Both are legitimate;
both need labeled sweeps.


In [ ]:
# RRF and normalized weighted fusion return document-index scores.
def rrf_scores(lexical, dense, k=60):
    total = np.zeros_like(lexical, dtype=np.float64)
    for scores in [lexical, dense]:
        for rank_position, index in enumerate(np.argsort(-scores, kind="stable"), start=1):
            total[index] += 1 / (k + rank_position)
    return total

def normalize(values):
    spread = values.max() - values.min()
    return np.zeros_like(values) if spread == 0 else (values - values.min()) / spread

def weighted_scores(lexical, dense, alpha=0.5):
    return (1 - alpha) * normalize(lexical) + alpha * normalize(dense)

def reciprocal_rank(scores, candidates, gold_ids):
    order = np.argsort(-scores, kind="stable")
    return next(
        (1 / position for position, index in enumerate(order, start=1)
         if candidates[index]["id"] in gold_ids),
        0.0,
    )


In [ ]:
# Sweep k and alpha on labeled queries; keep per-query evidence.
supported_cases = [case for case in RETRIEVAL_QUERIES if case["gold_ids"]]
sweep_rows = []
for case in supported_cases:
    candidates, lexical, dense = first_stage(case["question"], case["groups"])
    for k_value in [10, 60, 200]:
        sweep_rows.append({
            "query": case["id"], "family": "RRF", "setting": k_value,
            "mrr": reciprocal_rank(rrf_scores(lexical, dense, k_value), candidates, case["gold_ids"]),
        })
    for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
        sweep_rows.append({
            "query": case["id"], "family": "weighted", "setting": alpha,
            "mrr": reciprocal_rank(weighted_scores(lexical, dense, alpha), candidates, case["gold_ids"]),
        })
sweeps = pd.DataFrame(sweep_rows)
display(sweeps.groupby(["family", "setting"])["mrr"].mean().reset_index())


In [ ]:
# Static comparison: the average can tie while individual queries disagree.
pivot = sweeps.query("family == 'weighted'").pivot(index="query", columns="setting", values="mrr")
ax = pivot.plot(figsize=(9, 4), marker="o")
ax.set_title("Per-query MRR across weighted-fusion alpha")
ax.set_xlabel("query")
ax.set_ylabel("MRR")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## Part 3 - Rerank the shortlist, never the forbidden corpus

A pair reranker reads query-document features jointly. It may change order, not membership.
This tiny `nn.Module` exposes four features and fixed weights so you can inspect the forward
pass without downloading a cross-encoder.

![Reranking changes rank, not membership](images/cross-encoder-reranking-rank-movement.gif)


In [ ]:
# Pair features: normalized lexical score, dense score, code match, requested-detail match.
class PairReranker(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 1, bias=False)
        with torch.no_grad():
            self.linear.weight.copy_(torch.tensor([[0.35, 0.35, 0.20, 0.10]]))
        for parameter in self.parameters():
            parameter.requires_grad_(False)

    def forward(self, features):
        return self.linear(features).squeeze(-1)

reranker = PairReranker().to(DEVICE).eval()

def pair_features(query, document, lexical_value, dense_value):
    query_tokens, doc_tokens = set(tokenize(query)), set(tokenize(document["text"]))
    code_tokens = {token for token in query_tokens if any(character.isdigit() for character in token)}
    detail_tokens = query_tokens & {"weeks", "quotes", "days", "deductible"}
    return [
        lexical_value, dense_value,
        float(code_tokens <= doc_tokens),
        float(not detail_tokens or detail_tokens <= doc_tokens),
    ]


In [ ]:
# Measure rank movement on the exact-code case.
case = RETRIEVAL_QUERIES[0]
candidates, lexical, dense = first_stage(case["question"], case["groups"])
fused = weighted_scores(lexical, dense, alpha=0.75)
shortlist = np.argsort(-fused, kind="stable")[:3]
feature_rows = [
    pair_features(case["question"], candidates[index], normalize(lexical)[index], dense[index])
    for index in shortlist
]
with torch.no_grad():
    rerank_scores = reranker(torch.tensor(feature_rows, dtype=torch.float32, device=DEVICE)).cpu().numpy()
before = [candidates[index]["id"] for index in shortlist]
after = [before[index] for index in np.argsort(-rerank_scores, kind="stable")]
display(pd.DataFrame({"fused_order": before, "reranked_order": after}))
assert set(before) == set(after)
print("Membership preserved:", set(before) == set(after))


## Part 4 - Gate authorization before scoring and support before answering

The ACL gate is not a relevance feature. Missing identity, tenant, or group evaluation fails
closed. The support gate is separate: it can abstain even when retrieval has a topical winner.


In [ ]:
# Support evidence penalizes requested details absent from the top passage.
def support_score(query, document, lexical_score_value, dense_score_value):
    requested = set(tokenize(query)) & {"deductible", "dependants", "weeks", "quotes", "days"}
    missing = requested - set(tokenize(document["text"]))
    base = 0.35 * float(lexical_score_value > 0) + 0.65 * max(float(dense_score_value), 0.0)
    return max(0.0, min(1.0, base - 0.8 * bool(missing)))

def validate_identity(groups, tenant):
    if tenant != "riverside" or not groups:
        raise PermissionError("Identity or ACL context is incomplete; retrieval failed closed.")

unsupported = RETRIEVAL_QUERIES[-1]
validate_identity(unsupported["groups"], "riverside")
candidates, lexical, dense = first_stage(unsupported["question"], unsupported["groups"])
top_index = int(np.argmax(weighted_scores(lexical, dense)))
support = support_score(unsupported["question"], candidates[top_index], lexical[top_index], dense[top_index])
print(candidates[top_index]["title"], f"support={support:.3f}")
print("decision=ABSTAIN" if support < 0.45 else "decision=ANSWER")
assert support < 0.45


## Part 5 - Degraded mode is an observable state, not a quiet fallback

One retriever may fail. The surviving authorized path can serve a degraded result, but the
response and telemetry must say so. If both retrievers fail, abstain or error. ACL failures
never trigger a broader retry.


In [ ]:
# Execute one query with explicit failure injection and stage telemetry.
def retrieve_with_mode(case, fail_lexical=False, fail_dense=False):
    validate_identity(case["groups"], "riverside")
    started = time.perf_counter()
    candidates, lexical, dense = first_stage(case["question"], case["groups"])
    if fail_lexical and fail_dense:
        return [], {"mode": "unavailable", "decision": "ABSTAIN", "latency_ms": 0.0}
    if fail_lexical:
        scores, mode = dense, "dense-only"
    elif fail_dense:
        scores, mode = lexical, "lexical-only"
    else:
        scores, mode = rrf_scores(lexical, dense), "hybrid"
    order = np.argsort(-scores, kind="stable")[:3]
    elapsed = (time.perf_counter() - started) * 1000
    record = {
        "query_id": case["id"], "mode": mode, "decision": "RESULT",
        "authorized_candidates": len(candidates), "returned": len(order),
        "latency_ms": round(elapsed, 3),
    }
    return [candidates[index]["id"] for index in order], record

mode_rows = []
for flags in [(False, False), (True, False), (False, True), (True, True)]:
    _, record = retrieve_with_mode(RETRIEVAL_QUERIES[1], *flags)
    mode_rows.append(record)
telemetry = pd.DataFrame(mode_rows)
display(telemetry)
assert set(telemetry["mode"]) == {"hybrid", "dense-only", "lexical-only", "unavailable"}


In [ ]:
# Offline metrics and operational mode rates remain separate.
metric_rows = []
for case in supported_cases:
    ranked_ids, record = retrieve_with_mode(case)
    gold_rank = next(
        (position for position, doc_id in enumerate(ranked_ids, start=1) if doc_id in case["gold_ids"]),
        None,
    )
    metric_rows.append({
        "query": case["id"],
        "recall@3": float(bool(set(ranked_ids) & case["gold_ids"])),
        "mrr": 0.0 if gold_rank is None else 1 / gold_rank,
        "mode": record["mode"],
    })
metric_table = pd.DataFrame(metric_rows)
display(metric_table)
print(metric_table[["recall@3", "mrr"]].mean())


## Part 6 - Publish sparse and dense artifacts atomically

A manifest binds both indexes to one corpus revision. Checksums catch partial or mixed
publication. The previous complete version remains the rollback target.


In [ ]:
# Write artifacts only inside a temporary directory, then verify the manifest.
with tempfile.TemporaryDirectory() as directory:
    root = Path(directory)
    sparse_path = root / "sparse.json"
    dense_path = root / "dense.pt"
    sparse_path.write_text(json.dumps([tokenize(d["text"]) for d in DOCUMENTS]), encoding="utf-8")
    torch.save(document_embeddings.cpu(), dense_path)

    def checksum(path):
        return hashlib.sha256(path.read_bytes()).hexdigest()

    manifest = {
        "index_version": "riverside-policy-2026-10-06",
        "previous_complete_version": "riverside-policy-2026-09-28",
        "document_ids": [document["id"] for document in DOCUMENTS],
        "artifacts": {
            "sparse.json": checksum(sparse_path),
            "dense.pt": checksum(dense_path),
        },
    }
    manifest_path = root / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    loaded = json.loads(manifest_path.read_text(encoding="utf-8"))
    verified = all(checksum(root / name) == digest for name, digest in loaded["artifacts"].items())
    print(json.dumps(loaded, indent=2))
    print("manifest_verified=", verified)
    assert verified


## Lab scorecard

- **BM25:** exact identifiers remained visible.
- **PyTorch dense encoder:** paraphrases shared an explicit semantic feature space.
- **Fusion:** RRF and normalized weighted fusion were swept per query.
- **Reranking:** rank changed while candidate membership stayed fixed.
- **Security:** ACL filtering occurred before every scorer and failed closed.
- **Support:** a topical but unanswered deductible query abstained.
- **Resilience:** hybrid, lexical-only, dense-only, and unavailable modes were recorded.
- **Operations:** Recall@3/MRR stayed separate from mode telemetry.
- **Lineage:** sparse and dense checksums passed one immutable manifest.

**Your turn:** change one alpha or `k`, predict which query moves, then inspect the per-query
MRR table. Do not choose a winner from the aggregate alone.

Next, [RAG failure localization](02-rag-failure-localization-and-evaluation.ipynb) asks what
happens after retrieval: did the retriever miss, or did the generator misuse good evidence?
